In [11]:
!pip install -q \
  torch \
  peft \
  huggingface_hub \
  ipywidgets

!pip install -U datasets optuna

In [4]:
import os, shutil, gc, torch, optuna
from huggingface_hub import login, notebook_login, HfFolder, HfApi, hf_hub_download, delete_repo, list_repo_files, snapshot_download
os.environ["TRANSFORMERS_NO_TF"] = "1"   # prevents TF/Keras import
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig,
                          Trainer, TrainingArguments, DataCollatorWithPadding, default_data_collator)
from peft import (LoraConfig, get_peft_model, prepare_model_for_kbit_training)
from datasets import load_dataset, concatenate_datasets, Dataset, DatasetDict
import json
import inspect
import pandas as pd
import numpy as np
import tempfile
from datetime import datetime
from collections import Counter
from torch.utils.data import Dataset, Subset

In [5]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [6]:
api = HfApi()
login()

In [7]:
HF_DATASET_REPO = "eduhuemar001/news"

# Download JSON files from HF dataset repo
train_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename="train.json",
    repo_type="dataset",
    local_dir=".",
    local_dir_use_symlinks=False
)
eval_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename="eval.json",
    repo_type="dataset",
    local_dir=".",
    local_dir_use_symlinks=False
)

# Load datasets
with open(train_path, "r", encoding="utf-8") as f:
    dataset_train = json.load(f)

with open(eval_path, "r", encoding="utf-8") as f:
    dataset_eval = json.load(f)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:979: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


train.json:   0%|          | 0.00/70.9M [00:00<?, ?B/s]

eval.json:   0%|          | 0.00/23.1M [00:00<?, ?B/s]

In [8]:
class NewsPairDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        self.data = data                  # list of dicts with keys of title, text, status
        self.tok = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        item = self.data[i]
        title = (item.get("title") or "").strip()
        text  = (item.get("text")  or "").strip()
        label = int(item.get("status"))   # 0/1

        enc = self.tok(
            text=title,                   # News title
            text_pair=text,               # News article
            truncation="only_second",     # keep full title, truncate only news article
            max_length=self.max_length,
            padding=False,                # let collator pad
            return_attention_mask=True
        )

        out = {
            "input_ids": torch.tensor(enc["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(enc["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(label, dtype=torch.long),
        }
        return out

train_dataset = NewsPairDataset(dataset_train, tokenizer, max_length=512)
eval_dataset  = NewsPairDataset(dataset_eval,  tokenizer, max_length=512)

In [13]:
rns = np.random.RandomState(42)
TRAIN_CAP = min(len(train_dataset), 1000)
EVAL_CAP  = min(len(eval_dataset),  200)

train_idx = rns.choice(len(train_dataset), size=TRAIN_CAP, replace=False)
eval_idx  = rns.choice(len(eval_dataset),  size=EVAL_CAP,  replace=False)

train_small = Subset(train_dataset, sorted(train_idx))
eval_small  = Subset(eval_dataset,  sorted(eval_idx))

def objective(trial: optuna.Trial):
    # Hyperparameter search space
    hp = {
        "learning_rate":       trial.suggest_float("learning_rate", 1e-5, 5e-3, log=True),
        "weight_decay":        trial.suggest_float("weight_decay", 0.0, 0.2),
        "warmup_ratio":        trial.suggest_float("warmup_ratio", 0.0, 0.2),
        "lr_scheduler_type":   trial.suggest_categorical("lr_scheduler_type", ["linear", "cosine"]),
        "per_device_train_bs": trial.suggest_categorical("per_device_train_batch_size", [8, 16, 32]),
        "per_device_eval_bs":  trial.suggest_categorical("per_device_eval_batch_size",  [16, 32]),
        "gradient_accum":      trial.suggest_categorical("gradient_accumulation_steps", [1, 2, 4]),
        "num_train_epochs":    trial.suggest_int("num_train_epochs", 2, 6),
    }

    # ----- model -----
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    model = model.to("cuda")
    # print(next(model.parameters()).device)

    args = TrainingArguments(
        output_dir=f"./runs/trial_{trial.number}",
        learning_rate=hp["learning_rate"],
        weight_decay=hp["weight_decay"],
        warmup_ratio=hp["warmup_ratio"],
        lr_scheduler_type=hp["lr_scheduler_type"],
        per_device_train_batch_size=hp["per_device_train_bs"],
        per_device_eval_batch_size=hp["per_device_eval_bs"],
        gradient_accumulation_steps=hp["gradient_accum"],
        num_train_epochs=hp["num_train_epochs"],
        eval_strategy="steps",
        eval_steps=200,
        logging_strategy="steps",
        logging_steps=50,
        save_strategy="no",
        report_to="none",
        fp16=False,
        bf16=False,
        seed=42,
        load_best_model_at_end=False,
        dataloader_num_workers=0,
        optim="adamw_torch",
    )

    trainer = Trainer(
        model=model,
        args=args,
        tokenizer=tokenizer,
        train_dataset=train_small,
        eval_dataset=eval_small,
        data_collator=DataCollatorWithPadding(tokenizer),
    )

    trainer.train()
    metrics = trainer.evaluate()
    score = float(metrics["eval_loss"])

    # cleanup VRAM
    del trainer, model
    torch.cuda.empty_cache()

    return score

In [16]:
HF_REPO = "eduhuemar001/distilbert-news-tuning"
REMOTE_DB_FILE = "optuna_study.db"
LOCAL_DB = f"/content/{REMOTE_DB_FILE}"
STUDY_KEY = f"sqlite:///{LOCAL_DB}"

api = HfApi()

# Try to resume from HF Hub
try:
    db_path = hf_hub_download(
        repo_id=HF_REPO,
        filename=REMOTE_DB_FILE,
        repo_type="model",
        local_dir="/content",
        local_dir_use_symlinks=False
    )
    print(f"Resuming from HF DB at: {db_path}")
except Exception as e:
    print(f"No remote DB found (starting fresh): {e}")
    # Ensure local path exists, Optuna db be created
    if not os.path.exists(LOCAL_DB):
        open(LOCAL_DB, "wb").close()

# Create or resume study using the local SQLite file
study = optuna.create_study(
    study_name="distilbert_hpo",
    direction="minimize",
    storage=STUDY_KEY,
    load_if_exists=True
)

# 3) Callback: upload updated DB to HF after each trial
def upload_db_after_trial(study, trial):
    api.upload_file(
        path_or_fileobj=LOCAL_DB,
        path_in_repo=REMOTE_DB_FILE,
        repo_id=HF_REPO,
        repo_type="model",
        commit_message=f"Optuna DB update after trial {trial.number}"
    )
    print(f"Pushed DB after trial {trial.number}")

# 4) Run optimization (writes to SQLite every trial; then uploads)
study.optimize(objective, n_trials=20, callbacks=[upload_db_after_trial])

print("Best params:", study.best_trial.params)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:979: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(
[I 2025-11-09 18:19:17,771] Using an existing study with name 'distilbert_hpo' instead of creating a new one.


No remote DB found (starting fresh): 404 Client Error. (Request ID: Root=1-6910db25-2fcd86f43ff24aab3e124cd4;223ecd52-8d39-4b08-9a4e-e398c9b51d64)

Entry Not Found for url: https://huggingface.co/eduhuemar001/distilbert-news-tuning/resolve/main/optuna_study.db.


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-1510381963.py:53: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss,Validation Loss


[I 2025-11-09 18:24:11,743] Trial 4 finished with value: 0.00022262363927438855 and parameters: {'learning_rate': 0.0002111577160148881, 'weight_decay': 0.14170474728616472, 'warmup_ratio': 0.15983285501618702, 'lr_scheduler_type': 'cosine', 'per_device_train_batch_size': 8, 'per_device_eval_batch_size': 16, 'gradient_accumulation_steps': 4, 'num_train_epochs': 6}. Best is trial 4 with value: 0.00022262363927438855.


...

[I 2025-11-09 19:36:29,628] Trial 23 finished with value: 0.6871696710586548 and parameters: {'learning_rate': 0.0017078266244985958, 'weight_decay': 0.14580427496913684, 'warmup_ratio': 0.09051339748513235, 'lr_scheduler_type': 'cosine', 'per_device_train_batch_size': 8, 'per_device_eval_batch_size': 16, 'gradient_accumulation_steps': 4, 'num_train_epochs': 4}. Best is trial 4 with value: 0.00022262363927438855.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /content/optuna_study.db    : 100%|##########|  147kB /  147kB            

Pushed DB after trial 23
Best params: {'learning_rate': 0.0002111577160148881, 'weight_decay': 0.14170474728616472, 'warmup_ratio': 0.15983285501618702, 'lr_scheduler_type': 'cosine', 'per_device_train_batch_size': 8, 'per_device_eval_batch_size': 16, 'gradient_accumulation_steps': 4, 'num_train_epochs': 6}
